In [2]:
import pandas as pd          # For data manipulation and analysis
import numpy as np           # For numerical operations
import matplotlib.pyplot as plt  # For creating visualizations
import seaborn as sns        # For beautiful statistical charts
import os                    # For file path operations
import warnings
warnings.filterwarnings('ignore')  # Suppress unnecessary warnings

LOAD ALL DATASETS

In [3]:
orders    = pd.read_csv('olist_orders_dataset.csv')           # Main order info
items     = pd.read_csv('olist_order_items_dataset.csv')      # Items in each order
customers = pd.read_csv('olist_customers_dataset.csv')        # Customer details
products  = pd.read_csv('olist_products_dataset.csv')         # Product details
payments  = pd.read_csv('olist_order_payments_dataset.csv')   # Payment info
reviews   = pd.read_csv('olist_order_reviews_dataset.csv')    # Customer reviews
category  = pd.read_csv('product_category_name_translation.csv')  # Portuguese → English

In [4]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [5]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [6]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,02-10-2017 10:56,02-10-2017 11:07,04-10-2017 19:55,10-10-2017 21:25,18-10-2017 00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,24-07-2018 20:41,26-07-2018 03:24,26-07-2018 14:31,07-08-2018 15:27,13-08-2018 00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,08-08-2018 08:38,08-08-2018 08:55,08-08-2018 13:50,17-08-2018 18:06,04-09-2018 00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,18-11-2017 19:28,18-11-2017 19:45,22-11-2017 13:39,02-12-2017 00:28,15-12-2017 00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,13-02-2018 21:18,13-02-2018 22:20,14-02-2018 19:46,16-02-2018 18:17,26-02-2018 00:00


### DATA CLEANING 
 Fix date columns in Orders table
# These columns are stored as text (string/object) — we convert them to datetime -
# so we can do date calculations like delivery time

In [7]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], format='mixed', dayfirst=True)

In [9]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


#  Calculate how many days it took to deliver each order (new column)

In [10]:
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_purchase_timestamp']
).dt.days


# Keep only DELIVERED orders

We filter to only 'delivered' orders because:
# - Cancelled/returned orders don't have full delivery data
# - They would skew our revenue and delivery analysis

In [11]:
orders = orders[orders['order_status'] == 'delivered']
orders = orders.dropna(subset=['delivery_days'])  # Remove rows where delivery date is missing

In [12]:
products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [14]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

# Fix Products table nulls

 610 products have no category name — we fill with 'Unknown'
# instead of dropping, because these products ARE part of real orders

In [15]:
products['product_category_name']        = products['product_category_name'].fillna('Unknown')
products['product_name_lenght']          = products['product_name_lenght'].fillna(0)
products['product_description_lenght']   = products['product_description_lenght'].fillna(0)
products['product_photos_qty']           = products['product_photos_qty'].fillna(0)

In [ ]:
# For numeric measurements — fill with median

In [16]:
for col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    products[col] = products[col].fillna(products[col].median())


# MERGE ALL TABLES INTO ONE MASTER TABLE

In [ ]:
# We merged all tables using common keys (like order_id, customer_id, product_id)
# This creates one big master table with all information in one place


In [17]:
df = (
    orders
    .merge(items,     on='order_id',              how='left')  # Add item/price info
    .merge(customers, on='customer_id',           how='left')  # Add customer location
    .merge(products,  on='product_id',            how='left')  # Add product category
    .merge(payments,  on='order_id',              how='left')  # Add payment method
    .merge(category,  on='product_category_name', how='left')  # Add English category name
    .merge(reviews[['order_id', 'review_score']], on='order_id', how='left')  # Add review score
)

In [ ]:
# Create time-based columns for easier grouping in analysis

In [18]:
df['month'] = df['order_purchase_timestamp'].dt.to_period('M')  # e.g. 2017-11
df['year']  = df['order_purchase_timestamp'].dt.year             # e.g. 2017
df['month_name'] = df['order_purchase_timestamp'].dt.strftime('%b %Y')  # e.g. Nov 2017

In [19]:
df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  15
order_delivered_carrier_date        1
order_delivered_customer_date       0
order_estimated_delivery_date       0
delivery_days                       0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
product_category_name               0
product_name_lenght                 0
product_description_lenght          0
product_photos_qty                  0
product_weight_g                    0
product_length_cm                   0
product_height_cm                   0
product_widt

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115715 entries, 0 to 115714
Data columns (total 36 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       115715 non-null  object        
 1   customer_id                    115715 non-null  object        
 2   order_status                   115715 non-null  object        
 3   order_purchase_timestamp       115715 non-null  datetime64[ns]
 4   order_approved_at              115700 non-null  datetime64[ns]
 5   order_delivered_carrier_date   115714 non-null  datetime64[ns]
 6   order_delivered_customer_date  115715 non-null  datetime64[ns]
 7   order_estimated_delivery_date  115715 non-null  datetime64[ns]
 8   delivery_days                  115715 non-null  float64       
 9   order_item_id                  115715 non-null  int64         
 10  product_id                     115715 non-null  object        
 11  

# fill next nulls

In [21]:
df['product_category_name_english'] = df['product_category_name_english'].fillna('Unknown')

In [22]:
# Fill review score with median score
df['review_score'] = df['review_score'].fillna(df['review_score'].median())

In [25]:
df.isnull().sum()

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                15
order_delivered_carrier_date      1
order_delivered_customer_date     0
order_estimated_delivery_date     0
delivery_days                     0
order_item_id                     0
product_id                        0
seller_id                         0
shipping_limit_date               0
price                             0
freight_value                     0
customer_unique_id                0
customer_zip_code_prefix          0
customer_city                     0
customer_state                    0
product_category_name             0
product_name_lenght               0
product_description_lenght        0
product_photos_qty                0
product_weight_g                  0
product_length_cm                 0
product_height_cm                 0
product_width_cm                  0
payment_sequential          

In [29]:
median_approval = (df['order_approved_at'] - df['order_purchase_timestamp']).median()
df['order_approved_at'] = df['order_approved_at'].fillna(
    df['order_purchase_timestamp'] + median_approval)

median_carrier = (df['order_delivered_carrier_date'] - df['order_purchase_timestamp']).median()
df['order_delivered_carrier_date'] = df['order_delivered_carrier_date'].fillna(
    df['order_purchase_timestamp'] + median_carrier)


In [30]:
df.isnull().sum()


order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
delivery_days                    0
order_item_id                    0
product_id                       0
seller_id                        0
shipping_limit_date              0
price                            0
freight_value                    0
customer_unique_id               0
customer_zip_code_prefix         0
customer_city                    0
customer_state                   0
product_category_name            0
product_name_lenght              0
product_description_lenght       0
product_photos_qty               0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
payment_sequential               3
payment_type        

In [37]:
print(df[['payment_type', 'payment_sequential', 'payment_value']].isnull().sum())

payment_type          3
payment_sequential    3
payment_value         3
dtype: int64


In [38]:
# Only 3 rows have null payment info — safe to drop
df = df.dropna(subset=['payment_type', 'payment_sequential', 'payment_value'])

# Verify
print("Total nulls:", df.isnull().sum().sum())
print("Final shape:", df.shape)

Total nulls: 0
Final shape: (115712, 36)


# BUSINESS SUMMARY

In [39]:
Total_orders  = df['order_id'].nunique()
total_revenue = df['price'].sum()
avg_order_val = df.groupby('order_id')['price'].sum().mean()
avg_delivery  = df['delivery_days'].mean()
avg_review    = df['review_score'].mean()
 
print(f"   Total Orders     : {total_orders:>10,}")
print(f"   Total Revenue    : R$ {total_revenue:>12,.2f}")
print(f"   Avg Order Value  : R$ {avg_order_val:>10,.2f}")
print(f"   Avg Delivery Days: {avg_delivery:>10.1f} days")
print(f"   Avg Review Score : {avg_review:>10.2f} / 5.0")
 

   Total Orders     :     96,470
   Total Revenue    : R$ 13,875,087.58
   Avg Order Value  : R$     143.83
   Avg Delivery Days:       12.0 days
   Avg Review Score :       4.09 / 5.0


In [40]:
# Save final master table
df.to_csv('olist_master.csv', index=False)
print("✅ File saved! Ready for MySQL.")

✅ File saved! Ready for MySQL.
